## Feature Engineering

This notebook focuses on reading the raw per-sample JSON records and the manifest to extract the three feature domains
1. URL features: lexical features
2. HTML features: structural features, ratio-based
3. JavaScript AST features: inline script parsing only; using Esprima, pre-order traversal to produce note-type sequence tokens.

In [49]:
# library imports
import json
import logging
import math
import os
import re
import sys
import esprima
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse

In [27]:
# directory mapping
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
OUT_DIR = "data/features"
LOG_PATH = "logs/feature_engineering.log"

manifest_path = os.path.join(PROCESSED_DIR, "dataset_manifest.csv")
features_path = os.path.join(OUT_DIR, "features.csv")
js_seq_path = os.path.join(OUT_DIR, "js_node_sequences")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(js_seq_path, exist_ok=True)
os.makedirs("logs", exist_ok=True)

In [28]:
# logging configuration
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.flush = sys.stdout.flush
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), stream_handler],
    force=True,
)
log = logging.getLogger(__name__)

### Loading the dataset manifest

In [29]:
df = pd.read_csv(manifest_path)
log.info(
    f"Loaded manifest: {len(df)} rows "
    f"({(df["split"] == "train").sum()} train / {(df["split"] == "test").sum()} test)"
)
df.head()

2026-09-10 01:28:45,818 [INFO] Loaded manifest: 11211 rows (8968 train / 2243 test)


,sample_id,url,source,hostname,http_status,n_inline_scripts,n_external_scripts,html_length,label,split
0,a4c58a315f440f49,https://lnkz.at/UHNP7,phishtank,lnkz.at,200,2,4,107298,1,train
1,ece6b42a4122c1e4,https://www.bl.uk/,tranco,www.bl.uk,200,41,17,189312,0,train
2,01b65cd9d682e128,https://www.zndns.com/,tranco,www.zndns.com,200,1,6,25982,0,train
3,00104681a63fc3b9,http://0efbd9f.wcomhost.com/de/banking-postban...,phishing_db,0efbd9f.wcomhost.com,200,0,0,122,1,train
4,05d44ac34eee813c,https://liveball.sx/,tranco,liveball.sx,200,17,14,92198,0,train


### URL lexical features (F1-F12)

In [30]:
# IP address regular expressions
IP_RE = re.compile(r"^(?:\d{1,3}\.){3}\d{1,3}$")
IPV6_RE = re.compile(r"^\[?[0-9a-fA-F:]+\]?$")

In [31]:
def shannon_entropy(s):
    """Shanon entropy of a string, in bits. Empty string -> 0.0"""
    if not s:
        return 0.0
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in freq.values())

def is_ip_host(hostname):
    if not hostname:
        return False
    host = hostname.strip("[]")
    if IP_RE.match(host):
        return True
    if host.count(":") >= 2 and IPV6_RE.match(host):
        return True
    return False

def extract_url_features(record):
    url = record.get("seed_url") or ""
    parsed = urlparse(url if "://" in url else f"http://{url}")
    hostname = record.get("hostname") or parsed.hostname or ""
    labels = hostname.split(".") if hostname else []

    n = len(url)
    special_chars = sum(1 for ch in url if not ch.isalnum())

    return {
        "length": n,
        "dot_count": url.count("."),
        "hyphen_count": url.count("-"),
        "hyphen_count": url.count("-"),
        "hyphen_count": url.count("-"),
        "slash_count": url.count("/"),
        "digit_count": sum(1 for ch in url if ch.isdigit()),
        "has_ip_address": int(is_ip_host(hostname)),
        "has_https": int(parsed.scheme.lower() == "https"),
        "subdomain_count": max(0, len(labels) - 2),
        "query_length": len(parsed.query or ""),
        "entropy": round(shannon_entropy(url), 4),
        "special_char_ratio": round(special_chars / n, 6) if n else 0.0,
    }

### HTML structual features (F13-F27)

In [32]:
NULL_NREF_PATTERNS = {"#", "javascript:void(0)", "javascript:void(0);", "javascript:;"}
HIDDEN_STYLE_RE = re.compile(r"display\s*:\s*none|visibility\s*:\s*hidden", re.IGNORECASE)

In [39]:
def safe_ratio(n, d):
    return round(n / d, 6) if d else 0.0

def extract_html_features(html, hostname):
    if not html:
        return empty_html_features()
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception as e:
        return empty_html_features()

    a_tags = soup.find_all("a")
    a_with_href = [a for a in a_tags if a.get("href")]
    a_no_href = [a for a in a_tags if not a.get("href")]
    link_href = [t for t in soup.find_all("link") if t.get("href")]
    img_src = [t for t in soup.find_all("img") if t.get("src")]
    script_src = [t for t in soup.find_all("script") if t.get("src")]

    # F20
    total_hyperlinks = len(a_tags) + len(link_href) + len(img_src) + len(script_src)

    empty_hrefs = sum(
        1 for a in a_with_href
        if a["href"].strip().lower() in NULL_NREF_PATTERNS
    )

    internal, external = 0, 0
    for a in a_with_href:
        href_host = urlparse(a["href"]).hostname
        if not href_host:
            internal += 1
        elif hostname and href_host.lower().lstrip("www.") == hostname.lower().lstrip("www"):
            internal += 1
        else:
            external += 1

    # F25
    title_tag = soup.find("title")
    missing_title = int(title_tag is None or not title_tag.get_text(strip=True))

    hidden_count = 0
    for tag in soup.find_all(style=True):
        if HIDDEN_STYLE_RE.search(tag["style"]):
            hidden_count += 1

    for tag in soup.find_all(["script", "style"]):
        tag.decompose()
    visible_text_len = len(re.sub(r"\s+", " ", soup.get_text(separator=" ")).strip())
    all_tags_count = len(soup.find_all(True))

    return{
        "script_link_ratio": safe_ratio(len(script_src), total_hyperlinks),
        # F13
        "css_link_ratio": safe_ratio(len(link_href), total_hyperlinks),
        # F14
        "image_link_ratio": safe_ratio(len(img_src), total_hyperlinks),
        # F15
        "anchor_link_ratio": safe_ratio(len(a_with_href), total_hyperlinks),
        # F16
        "empty_link_ratio": safe_ratio(empty_hrefs, total_hyperlinks),
        # F17
        "anchor_without_href_ratio": safe_ratio(len(a_no_href), total_hyperlinks),
        # F19
        "total_hyperlinks": total_hyperlinks,
        # F20
        "internal_link_ratio": safe_ratio(internal, total_hyperlinks),
        # F21
        "external_link_ratio": safe_ratio(external, total_hyperlinks),
        # F22
        "external_internal_ratio": safe_ratio(external, internal),
        # F23
        "total_forms": len(soup.find_all("form")),
        # F24
        "missing_title": missing_title,
        # F25
        "hidden_elements_count": hidden_count,
        # F26
        "text_to_tag_ratio": safe_ratio(visible_text_len, all_tags_count),
        }

def empty_html_features():
    return {
        "script_link_ratio": 0.0, "css_link_ratio": 0.0, "image_link_ratio": 0.0,
        "anchor_link_ratio": 0.0, "empty_link_ratio": 0.0, "anchor_without_href_ratio": 0.0, "total_hyperlinks": 0,
        "internal_link_ratio": 0.0, "external_link_ratio": 0.0,
        "external_internal_ratio": 0.0, "total_forms": 0, "missing_title": 1,
        "hidden_elements_count": 0, "text_to_tag_ratio": 0.0,
    }

### JavaScript AST node-type sequences

In [41]:
def traverse_ast(node, out):
    """Pre-order traversal grammatical 'type' tokens"""
    if isinstance(node, dict):
        t = node.get("type")
        if t:
            out.append(t)
        for k, v in node.items():
            if k == "type":
                continue
            traverse_ast(v, out)
    elif isinstance(node, list):
        for item in node:
            traverse_ast(item, out)

def extract_js_node_sequence(inline_scripts):
    """Returns token sequence list, n_parsed, and n_failed
       for a sample's inline scripts"""
    inline_scripts = [s for s in (inline_scripts or []) if s and s.strip()]
    all_tokens = []
    n_parsed, n_failed = 0, 0

    for script in inline_scripts:
        tree_dict = None
        try:
            tree_dict = esprima.parseScript(script, tolerant=True).toDict()
        except Exception:
            try:
                tree_dict = esprima.parseModule(script, tolerant=True).toDict()
            except Exception:
                n_failed += 1
                continue
        tokens = []
        traverse_ast(tree_dict, tokens)
        all_tokens.extend(tokens)
        n_parsed += 1

    return all_tokens, n_parsed, n_failed

### Extract features

In [43]:
progress_every = 2000  # report progress after every 2000 processed samples
rows, n_missing_json, n_corrupt = [], 0, 0

for i, rec_row in enumerate(df.itertuples(), start=1):
    json_path = os.path.join(RAW_DIR, f"{rec_row.sample_id}.json")

    if not os.path.exists(json_path):
        log.warning(f"[missing] no raw json for sample_id={rec_row.sample_id} --- Dropped")
        n_missing_json += 1
        continue

    try:
        with open (json_path, "r", encoding="utf-8") as f:
            record = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
        log.warning(f"[corrupt] {json_path} could not parse -> {e}")
        n_corrupt += 1
        continue

    url_feats = extract_url_features(record)
    html_feats = extract_html_features(record.get("raw_html"), record.get("hostname"))
    js_tokens, n_parsed, n_failed = extract_js_node_sequence(record.get("inline_scripts"))

    row = {
        "sample_id": rec_row.sample_id,
        "label": rec_row.label,
        "split": rec_row.split,
        "source": rec_row.source,
        "hostname": rec_row.hostname,
    }

    row.update({f"url_{k}": v for k, v in url_feats.items()})
    row.update({f"html_{k}": v for k, v in html_feats.items()})
    row["js_n_inline_scripts"] = len(record.get("inline_scripts") or [])
    row["js_n_parsed_scripts"] = n_parsed
    row["js_n_failed_scripts"] = n_failed
    row["js_node_sequence_length"] = len(js_tokens)
    rows.append(row)

    seq_path = os.path.join(js_seq_path, f"{rec_row.sample_id}.txt")
    with open (seq_path, "w", encoding="utf-8") as f:
        f.write(" ".join(js_tokens))

    if progress_every and i % progress_every == 0:
        log.info(f"Processed {i} / {len(df)} manifest rows")

log.info(
    f"Featurised {len(rows)} samples "
    f"({n_missing_json} mising raw JSON, {n_corrupt} corrupt discarded)"
)

c:\Users\G22L1730\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\esprima\scanner.py:1040: FutureWarning: Possible nested set at position 1
  return re.compile(pattern, pyflags)
C:\Users\G22L1730\AppData\Local\Temp\ipykernel_9212\565837453.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSou

2026-09-10 01:34:50,453 [INFO] Processed 2000 / 11211 manifest rows
2026-09-10 01:36:49,999 [INFO] Processed 4000 / 11211 manifest rows
2026-09-10 01:38:50,551 [INFO] Processed 6000 / 11211 manifest rows
2026-09-10 01:40:52,781 [INFO] Processed 8000 / 11211 manifest rows
2026-09-10 01:42:51,965 [INFO] Processed 10000 / 11211 manifest rows


c:\Users\G22L1730\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\esprima\scanner.py:1040: FutureWarning: Possible set difference at position 11
  return re.compile(pattern, pyflags)
c:\Users\G22L1730\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\esprima\scanner.py:1040: FutureWarning: Possible set difference at position 22
  return re.compile(pattern, pyflags)


2026-09-10 01:44:00,101 [INFO] Featurised 11211 samples (0 mising raw JSON, 0 corrupt discarded)


In [48]:
features_df = pd.DataFrame(rows)
features_df.to_csv(features_path, index=False)
log.info(f"Wrote engineered features -> {OUT_DIR} ({len(features_df)} rows)")
log.info(f"JS AST node-type sequences (pre-BPE) written to {js_seq_path} / for TF-IDF fitting at model time.")
features_df.head()

2026-09-10 01:49:51,264 [INFO] Wrote engineered features -> data/features (11211 rows)
2026-09-10 01:49:51,264 [INFO] JS AST node-type sequences (pre-BPE) written to data/features\js_node_sequences / for TF-IDF fitting at model time.


,sample_id,label,split,source,hostname,url_length,url_dot_count,url_hyphen_count,url_slash_count,url_digit_count,...,html_external_link_ratio,html_external_internal_ratio,html_total_forms,html_missing_title,html_hidden_elements_count,html_text_to_tag_ratio,js_n_inline_scripts,js_n_parsed_scripts,js_n_failed_scripts,js_node_sequence_length
0,a4c58a315f440f49,1,train,phishtank,lnkz.at,21,1,0,3,1,...,0.000000,0.000000,0,0,1,1.292453,2,2,0,38
1,ece6b42a4122c1e4,0,train,tranco,www.bl.uk,13,1,0,2,0,...,0.195652,0.428571,0,0,1,4.072435,41,41,0,735
2,01b65cd9d682e128,0,train,tranco,www.zndns.com,17,1,0,2,0,...,0.210526,0.413793,1,0,0,7.590659,1,1,0,275
3,00104681a63fc3b9,1,train,phishing_db,0efbd9f.wcomhost.com,98,2,1,9,15,...,0.000000,0.000000,0,1,0,43.500000,0,0,0,0
4,05d44ac34eee813c,0,train,tranco,liveball.sx,19,1,0,2,0,...,0.003876,0.009804,1,0,0,5.140988,17,17,0,2088
